# ETL exploration

A walkthrough of the transformations in `etl.py` on a single song file and a single log file, to inspect the raw data and check each step before running the full pipeline.

In [1]:
import glob
import os

import pandas as pd

import etl

## Song data

Each song file holds one JSON record describing a song and its artist.

In [2]:
song_files = etl.get_files(etl.SONG_DATA_DIR)
print(len(song_files), 'song files')
song_df = pd.read_json(song_files[0], lines=True)
song_df.T

71 song files


,0
num_songs,1
artist_id,ARD7TVE1187B99BFB1
artist_latitude,NaN
artist_longitude,NaN
artist_location,California - LA
artist_name,Casual
song_id,SOMZWCG12A8C13C480
title,I Didn't Mean To
duration,218.93179
year,0


### `songs` and `artists` rows

A `year` of 0 means the year is unknown and is stored as `NULL`.

In [3]:
song_df[['song_id', 'title', 'artist_id', 'year', 'duration']]

,song_id,title,artist_id,year,duration
0,SOMZWCG12A8C13C480,I Didn't Mean To,ARD7TVE1187B99BFB1,0,218.93179


In [4]:
song_df[['artist_id', 'artist_name', 'artist_location', 'artist_latitude', 'artist_longitude']]

,artist_id,artist_name,artist_location,artist_latitude,artist_longitude
0,ARD7TVE1187B99BFB1,Casual,California - LA,NaN,NaN


## Log data

Each log file holds one day of user activity events. Only `NextSong` events represent a song being played.

In [5]:
log_files = etl.get_files(etl.LOG_DATA_DIR)
print(len(log_files), 'log files')
log_df = pd.read_json(log_files[0], lines=True)
log_df['page'].value_counts()

30 log files


page
NextSong    11
Home         3
Upgrade      1
Name: count, dtype: int64

In [6]:
plays = log_df[log_df['page'] == 'NextSong'].copy()
plays.head()

,artist,auth,firstName,gender,itemInSession,lastName,length,level,location,method,page,registration,sessionId,song,status,ts,userAgent,userId
2,Des'ree,Logged In,Kaylee,F,1,Summers,246.30812,free,"Phoenix-Mesa-Scottsdale, AZ",PUT,NextSong,1540344794796,139,You Gotta Be,200,1541106106796,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",8
4,Mr Oizo,Logged In,Kaylee,F,3,Summers,144.03873,free,"Phoenix-Mesa-Scottsdale, AZ",PUT,NextSong,1540344794796,139,Flat 55,200,1541106352796,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",8
5,Tamba Trio,Logged In,Kaylee,F,4,Summers,177.18812,free,"Phoenix-Mesa-Scottsdale, AZ",PUT,NextSong,1540344794796,139,Quem Quiser Encontrar O Amor,200,1541106496796,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",8
6,The Mars Volta,Logged In,Kaylee,F,5,Summers,380.42077,free,"Phoenix-Mesa-Scottsdale, AZ",PUT,NextSong,1540344794796,139,Eriatarka,200,1541106673796,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",8
7,Infected Mushroom,Logged In,Kaylee,F,6,Summers,440.26730,free,"Phoenix-Mesa-Scottsdale, AZ",PUT,NextSong,1540344794796,139,Becoming Insane,200,1541107053796,"""Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebK...",8


### Timestamps

`ts` is a Unix epoch in **milliseconds**, so it must be parsed with `unit='ms'`.

In [7]:
plays['start_time'] = pd.to_datetime(plays['ts'], unit='ms')
t = plays['start_time']
pd.DataFrame({
    'start_time': t,
    'hour': t.dt.hour,
    'day': t.dt.day,
    'week': t.dt.isocalendar().week,
    'month': t.dt.month,
    'year': t.dt.year,
    'weekday': t.dt.day_of_week,
}).head()

,start_time,hour,day,week,month,year,weekday
2,2018-11-01 21:01:46.796,21,1,44,11,2018,3
4,2018-11-01 21:05:52.796,21,1,44,11,2018,3
5,2018-11-01 21:08:16.796,21,1,44,11,2018,3
6,2018-11-01 21:11:13.796,21,1,44,11,2018,3
7,2018-11-01 21:17:33.796,21,1,44,11,2018,3


### Users

A user can upgrade from `free` to `paid` during the day, so the latest record per user is kept.

In [8]:
(plays.sort_values('start_time')
      [['userId', 'firstName', 'lastName', 'gender', 'level']]
      .drop_duplicates('userId', keep='last')
      .head())

,userId,firstName,lastName,gender,level
9,8,Kaylee,Summers,F,free
10,10,Sylvie,Cruz,F,free
13,26,Ryan,Smith,M,free
14,101,Jayden,Fox,M,free


### Matching events to songs

Log events only carry the song title, artist name and length. `etl.py` resolves them to `song_id` / `artist_id` with a lookup built from the `songs` and `artists` tables. Because the song dataset is a small sample, very few events find a match.

In [9]:
songs = pd.concat(pd.read_json(f, lines=True) for f in song_files)
known = set(zip(songs['title'], songs['artist_name']))
all_plays = pd.concat(pd.read_json(f, lines=True) for f in log_files)
all_plays = all_plays[all_plays['page'] == 'NextSong']
matched = sum((s, a) in known for s, a in zip(all_plays['song'], all_plays['artist']))
print(f'{matched} of {len(all_plays)} song plays match a song in the catalogue')

1 of 6820 song plays match a song in the catalogue
